In [10]:
import awkward as ak
import uproot
import numpy as np
import pandas as pd
import os
import json
import time

from coffea import processor
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema

import sys
sys.path.append("..")
from src.processors.Processor import Processor

# Dataset

In [2]:
base = '/data/pubfs/fudawei/mc'
basedir = {d: os.path.join(base, d) for d in os.listdir(base) if d=='GJets' or d=='ZpToHGamma'}
basedir

{'GJets': '/data/pubfs/fudawei/mc/GJets',
 'ZpToHGamma': '/data/pubfs/fudawei/mc/ZpToHGamma'}

In [3]:
samples = {s: [] for s in basedir}
for s in basedir:
    for (current_path, dirs, files) in os.walk(basedir[s]):
        for f in files:
            if f.endswith('.root'):
                samples[s].append(os.path.join(current_path, f))
samples

{'GJets': ['/data/pubfs/fudawei/mc/GJets/HT-40To100/3B93385F-77FD-6B47-BF83-8476E32C7515.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/B7ED4838-B32C-E347-BD38-0FE7F990A127.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/42A269A9-5AB5-1045-B806-EE5D4588043B.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/5C460723-1125-3540-8FDE-472EF4ED064C.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/A7BEFB69-C329-C445-AA66-6758B5AD5CF4.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/59666184-B2AA-464C-8946-7384CB7F3626.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/7B14B631-E17C-FF41-B1B1-8AF5F9EE69D0.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/E4DB23B8-2ECA-2147-A8F1-BB5C1FCBCCC4.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/ED0DC548-72E5-D146-BD08-CC494A7BE30B.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/1D64B0C2-65B4-BE4D-9478-4ED80332AB41.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/070AD8C7-183E-EE41-BB0F-04B720E8FBAB.root',
  '/data/pubfs/fudawei/mc/GJets/HT

# Batch test

In [4]:
_events = NanoEventsFactory.from_root(file=samples['ZpToHGamma'][0], treepath='Events', schemaclass=NanoAODSchema,).events()
p=Processor(outdir='../output/test/')
p.process(_events)

/home/pku/fudawei/.local/lib/python3.8/site-packages/awkward/operations/convert.py:1960: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if distutils.version.LooseVersion(
/home/pku/fudawei/.local/lib/python3.8/site-packages/awkward/operations/convert.py:1962: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  ) < distutils.version.LooseVersion("2.0.0"):


{'raw': 50000,
 'trigger': 44423,
 'b-veto': 22974,
 'muon': 21345,
 'electron': 19680,
 'photon': 17238,
 'AK8jet': 10053,
 'photon-jet_cleaning': 9763}

In [5]:
p.variables

{'AK8jet_mass': <Array [[42.1], [120], ... [85.9], [106]] type='9763 * option[var * float32[para...'>,
 'AK8jet_msoftdrop': <Array [[31.5], [117], ... [52], [49]] type='9763 * option[var * float32[paramet...'>,
 'AK8jet_eta': <Array [[0.466], [0.204], ... -0.168], [-1.07]] type='9763 * option[var * float3...'>,
 'AK8jet_pt': <Array [[322], [408], ... [368], [502]] type='9763 * option[var * float32[parame...'>,
 'AK8jet_phi': <Array [[3.06], [-2.88], ... [0.953], [2.48]] type='9763 * option[var * float32[...'>,
 'photon_eta': <Array [[-1.23], [1.28], ... -0.052], [-0.659]] type='9763 * option[var * float3...'>,
 'photon_phi': <Array [[-0.0207], [0.199, ... 2.99], [-0.664]] type='9763 * option[var * float3...'>,
 'photon_mass': <Array [[0], [0], [0], ... [0], [0], [0]] type='9763 * option[var * float32]'>,
 'photon_pt': <Array [[339], [426], ... [636], [471]] type='9763 * option[var * float32[parame...'>,
 'event_MET_pt': <Array [69.4, 23.5, 63.5, ... 76, 181, 32.4] type='9763 * ?float32

# Parallel computing

In [11]:
cutflow = {}
t0 = time.time()

In [12]:
def parallelProcess(samples, name):
    global cutflow, t0
    cutflow[name] = processor.run_uproot_job(
        fileset={name: samples[name]},
        treename="Events",
        processor_instance=Processor(outdir=f'../output/{name}'),
        executor=processor.futures_executor,
        executor_args={"schema": NanoAODSchema, "workers": 48}, # running on $workers cpu cores
    )
    cutflow['time_'+name] = (time.time() - t0)/60
    with open('../output/cutflow.txt', 'w') as file:
        json.dump(cutflow, file)

parallelProcess(samples=samples, name='ZpToHGamma')
parallelProcess(samples=samples, name='GJets')
#parallelProcess(samples=samples, name='WJetsToQQ')
#parallelProcess(samples=samples, name='ZJetsToQQ')
#parallelProcess(samples=samples, name='QCD')



Processing:   0%|          | 0/50 [00:00<?, ?chunk/s]

/home/pku/fudawei/.local/lib/python3.8/site-packages/awkward/operations/convert.py:1960: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if distutils.version.LooseVersion(
/home/pku/fudawei/.local/lib/python3.8/site-packages/awkward/operations/convert.py:1962: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  ) < distutils.version.LooseVersion("2.0.0"):
/home/pku/fudawei/.local/lib/python3.8/site-packages/awkward/operations/convert.py:1960: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if distutils.version.LooseVersion(
/home/pku/fudawei/.local/lib/python3.8/site-packages/awkward/operations/convert.py:1962: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  ) < distutils.version.LooseVersion("2.0.0"):
/home/pku/fudawei/.local/lib/python3.8/site-packages/awkward/operations/convert.py:1960: DeprecationWarn

Preprocessing:   0%|          | 0/147 [00:00<?, ?file/s]

Processing:   0%|          | 0/521 [00:00<?, ?chunk/s]

/home/pku/fudawei/.local/lib/python3.8/site-packages/coffea/processor/executor.py:249: UserWarning: Early stop: cancelled 302 jobs, will wait for 97 running jobs to complete
  warnings.warn(
/home/pku/fudawei/.local/lib/python3.8/site-packages/coffea/processor/executor.py:249: UserWarning: Early stop: cancelled 0 jobs, will wait for 96 running jobs to complete
  warnings.warn(
/home/pku/fudawei/.local/lib/python3.8/site-packages/coffea/processor/executor.py:249: UserWarning: Early stop: cancelled 0 jobs, will wait for 95 running jobs to complete
  warnings.warn(
/home/pku/fudawei/.local/lib/python3.8/site-packages/coffea/processor/executor.py:249: UserWarning: Early stop: cancelled 0 jobs, will wait for 93 running jobs to complete
  warnings.warn(
/home/pku/fudawei/.local/lib/python3.8/site-packages/coffea/processor/executor.py:249: UserWarning: Early stop: cancelled 0 jobs, will wait for 92 running jobs to complete
  warnings.warn(
/home/pku/fudawei/.local/lib/python3.8/site-packages/